# Pattern 3: Reflection / Critic Loop

This notebook demonstrates generate -> critique -> revise cycles to improve quality.

In [ ]:
from typing import TypedDict

from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph

MODEL_NAME = "qwen3.5:2b"
llm = ChatOllama(model=MODEL_NAME, temperature=0.2)

In [ ]:
MAX_ITERS = 2

class ReflectionState(TypedDict):
    prompt: str
    draft: str
    feedback: str
    iteration: int

def draft_node(state: ReflectionState) -> ReflectionState:
    if state['iteration'] == 0 and not state['draft']:
        d = llm.invoke(f"Write a clear answer:\n{state['prompt']}").content
        state['draft'] = str(d)
    else:
        revise_prompt = f"""
Revise this draft using feedback.
Draft:
{state['draft']}

Feedback:
{state['feedback']}
"""
        state['draft'] = str(llm.invoke(revise_prompt).content)
    return state

def critic_node(state: ReflectionState) -> ReflectionState:
    critique_prompt = f"""
Critique the following draft in 3 bullets: correctness, clarity, completeness.
Draft:
{state['draft']}
"""
    state['feedback'] = str(llm.invoke(critique_prompt).content)
    state['iteration'] += 1
    return state

def router(state: ReflectionState) -> str:
    if state['iteration'] >= MAX_ITERS:
        return 'finish'
    return 'improve'

In [ ]:
builder = StateGraph(ReflectionState)
builder.add_node('draft', draft_node)
builder.add_node('critic', critic_node)

builder.add_edge(START, 'draft')
builder.add_edge('draft', 'critic')
builder.add_conditional_edges('critic', router, {
    'improve': 'draft',
    'finish': END,
})

graph = builder.compile()

initial_state = {
    'prompt': 'Write a practical explanation of what an AI agent is for a beginner.',
    'draft': '',
    'feedback': '',
    'iteration': 0,
}

out = graph.invoke(initial_state)
out['draft']

## Experiment
Increase `MAX_ITERS` to see stronger iterative refinement, at the cost of latency.